# MLflow Tracing Demo

Run the real Virtue Foundation agent on five Ghana facility rows, then inspect MLflow or the JSONL fallback trace. The run structure is parent facility run -> five agent steps with citations.

In [ ]:
import asyncio, csv, json
from pathlib import Path

from src.agents.graph import run_agent

rows = list(csv.DictReader(Path('data/ghana_facilities.csv').open(newline='', encoding='utf-8')))
states = [asyncio.run(run_agent(row)) for row in rows[:5]]
[(state['source_row_id'], state.get('mlflow_run_id')) for state in states]

## MLflow URL

If MLflow is installed and running locally, open http://localhost:5000. Without MLflow, inspect `artifacts/mlflow_trace_fallback.jsonl`.

In [ ]:
fallback = Path('artifacts/mlflow_trace_fallback.jsonl')
events = [json.loads(line) for line in fallback.read_text(encoding='utf-8').splitlines()] if fallback.exists() else []
events[-15:]

In [ ]:
import pandas as pd

step_events = [event for event in events if event.get('event') == 'step']
citation_counts = pd.DataFrame([
    {'step': event['step'], 'citations': len(set(event.get('citations', [])))}
    for event in step_events
])
citation_counts.groupby('step')['citations'].sum().plot(kind='bar', title='Citations per agent step')

In [ ]:
states[0].get('citations', {})